# Semana 4 - Pipeline Final + ML

**Carga horaria:** 10h  
**Entregavel:** Dataset pronto para modelo

## Objetivos
- Organizar fluxo final de preparacao para modelagem.
- Separar treino e teste de forma reprodutivel.
- Treinar um baseline de classificacao para validar pipeline.

In [ ]:
from pathlib import Path
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

In [ ]:
input_file = Path('data/iris_features.csv')
if not input_file.exists():
    raise FileNotFoundError('Execute primeiro o notebook da Semana 3 para gerar data/iris_features.csv')

df = pd.read_csv(input_file)
print('Shape de entrada:', df.shape)
display(df.head())

## Definicao de alvo e atributos
- Alvo: `species_name`
- Atributos: todas as colunas, exceto `species_name`

In [ ]:
target_col = 'species_name'
X = df.drop(columns=[target_col])
y = df[target_col]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print('Treino:', X_train.shape, y_train.shape)
print('Teste :', X_test.shape, y_test.shape)

In [ ]:
numeric_cols = X_train.select_dtypes(include=['number']).columns.tolist()
categorical_cols = X_train.select_dtypes(exclude=['number']).columns.tolist()

numeric_pipeline = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_pipeline = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_pipeline, numeric_cols),
        ('cat', categorical_pipeline, categorical_cols)
    ]
)

model = Pipeline(steps=[
    ('prep', preprocessor),
    ('clf', LogisticRegression(max_iter=1000))
])

model.fit(X_train, y_train)
pred = model.predict(X_test)

acc = accuracy_score(y_test, pred)
print(f'Accuracy: {acc:.4f}')
print(classification_report(y_test, pred))

In [ ]:
# Exportar dados preparados para etapas futuras
final_dir = Path('data/final')
final_dir.mkdir(parents=True, exist_ok=True)

train_df = X_train.copy()
train_df[target_col] = y_train.values

test_df = X_test.copy()
test_df[target_col] = y_test.values

train_df.to_csv(final_dir / 'train_dataset.csv', index=False)
test_df.to_csv(final_dir / 'test_dataset.csv', index=False)

print('Arquivos gerados em', final_dir)
print('- train_dataset.csv')
print('- test_dataset.csv')

In [ ]:
# Validacao do entregavel
assert (Path('data/final/train_dataset.csv')).exists()
assert (Path('data/final/test_dataset.csv')).exists()
assert acc >= 0.7, 'Acuracia muito baixa para baseline do Iris.'
print('Entregavel Semana 4 OK: dataset pronto para modelo.')